# 28 (MLA) — Model Persistence

**ML Analyst perspective.** Train a model once, save it to IRIS, and reload it later — the model survives the session. Persistence stores the fitted state (coefficients, scaler stats, pipeline stages) as JSON in an IRIS table, and the planner/metadata (backend, LogicalVector) survives the round-trip.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Load the prepared splits

The `mla_train`/`mla_test` tables created in notebook 24.

In [ ]:
try:
    train = session.table("mla_train")
    test = session.table("mla_test")
    print("train:", train.count(), "| test:", test.count())
except Exception:
    print("Run notebook 24 first to create mla_train / mla_test.")

## 2. Train a model

A `LogisticRegression` on the standardized credit features.

In [ ]:
from irispark.ml.classification import LogisticRegression

feats = ["renda_std", "idade_std", "historico_idx_std", "divida_ratio_std"]
lr = LogisticRegression(featuresCol=feats, labelCol="inadimplente", maxIter=500, learningRate=0.5, regParam=0.1)
model = lr.fit(train)
print("trained; backend:", model.backend)

## 3. Save the model

`model.save(name, session=session)` persists it to IRIS and returns a model id.

In [ ]:
from irispark.ml import save, load, load_by_name, list_models, delete_model

mid = model.save("mla_default_model", session=session)
print("saved model id:", mid)

## 4. List saved models

In [ ]:
for m in list_models(session):
    print(m)

## 5. Load it back and predict

`load(id, session)` reconstructs the fitted model — coefficients survive.

In [ ]:
loaded = load(mid, session)
print("coefficients match:", loaded.coefficients == model.coefficients)
pred = loaded.transform(test)
pred.select("cliente_id", "inadimplente", "prediction").show(5)

## 6. Load by name

Don't have the id? Reload the most recent model with a given name.

In [ ]:
loaded2 = load_by_name("mla_default_model", session)
print("reloaded by name, coeffs match:", loaded2.coefficients == model.coefficients)

## 7. Persist a full pipeline

A `PipelineModel` (transformer → transformer → estimator) persists its nested stages and reloads as a single predict-ready object.

In [ ]:
from irispark.ml.feature import StandardScaler, VectorAssembler
from irispark.ml.pipeline import Pipeline

pipe = Pipeline(stages=[
    StandardScaler(inputCol="renda_imp", outputCol="renda_s"),
    VectorAssembler(inputCols=["renda_s", "idade", "historico_idx", "divida_ratio"], outputCol="features"),
    LogisticRegression(featuresCol=["renda_s", "idade", "historico_idx", "divida_ratio"], labelCol="inadimplente"),
])
pm = pipe.fit(train)
pmid = pm.save("mla_pipeline_model", session=session)
loaded_pm = load(pmid, session)
print("pipeline stages after reload:", len(loaded_pm.getStages()))
pred_pm = loaded_pm.transform(test).to_pandas()
print("pipeline predicts rows:", len(pred_pm))

## 8. Metadata survives the round-trip

The planner `backend` and the `LogicalVector` wiring persist with the model.

In [ ]:
print("loaded backend:", loaded.backend)
print("loaded logicalVector:", loaded.logicalVector)

## 9. Delete

Clean up so repeat runs stay tidy.

In [ ]:
delete_model(mid, session)
delete_model(pmid, session)
print("remaining models:", list_models(session))

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")